# Machine Learning (ML) separate pre-processes

This notebook is a customized project for separate pre-processing of indicators, covariates and spectra from the EU funded project [AI4SoilHealth](https://ai4soilhealth.eu). The project requires customization of the xspatula core framework. By default the path to the start module for a custom project is ./src/"project_name"; it needs to contain a startup function. The default name of that function is "Run_process". 

The aim of the pre-processing is to remove indicator outlier, and remove physical artifacts, instrument noise and baseline shifts from spectral data.

## Separate versus compiled pre-processes

The pre-processes defined in this notebook operate separately on indicators, covariates and spectra. Neither the relation between covariates or spectra on the one hand and the target indicators on the other, nor the regressor applied, are considered. The methods below are faster to apply and meant as an initial clean-up, data compression and information enhancement stage. With compiled data you can apply more advanced data compression and information enhancement that consider both the relation between covariates/spectra and the target indicators with the notebook `ml_preprocess_compiled`.

## How to use this notebook

The idea with this notebook is to allow you to test and understand different pre-processing options, evaluate the results and then define a pre-processing pipeline that fits your need. To create a pre-processing pipeline you stack sequential pre-processes together by using the output from an existing result (or the raw input data for the first pipeline step) as input to the next. To keep track of the outputs, every pre-process adds a suffix to the name of its input dataframe (e.g. `1350-2500_10.snv.d1`) and saves the full pre-processing chain in a companion JSON file. The suffix added by each pre-process is given at the end of its cell. Pre-processes with settings that the suffix does not show (outlier detection, filters, variance threshold, ward clustering and the chemometric chain) also accept the parameter "name_extension" (1-6 letters, digits or -) that is added after a hashtag, e.g. `.ma5#12321` for a moving average with the kernel weights [1, 2, 3, 2, 1]. See cell 2.1.5 for an example.

For detailed instructions on this notebook, please consult the online manual for [TO BE DONE](#).

In [ ]:
# Standard library imports
from os import path, getcwd

import sys 

# Get the working directory of the notebook
notebook_path = getcwd()

# Set the path to the root directory of this jupyter notebook (python) project
module_path = path.abspath(path.join('../..'))

if module_path not in sys.path:
    
    sys.path.append(module_path)

# Package application imports
from src.lib import Initiate_process

# Import the custom class for running processes defined in the ai4sh package
from src.ai4sh import Run_process

## Scheme file
To run this cell you need to set the [scheme_file](https://xspatula.github.io/setup_core_db_docs/framework/scheme_file/), that defines the location (path) to the project and associated process files and your user. You can set the path to the scheme file with a relative path (relative to this notebook), in your home directory using tilde (~) or with an absolute path in the syntax of your operating system. 

In [ ]:
# Set the path to the scheme_file  

scheme_file = '../scheme_ai4sh.json'

# 1 Indicator pre-processing

Indicator pre-processing sets single indicator values to missing (NaN), never whole samples: removing outliers (1.1) and removing values outside a range (1.2).

## 1.1 Detect and remove indicator outliers

Detect and select to remove outliers for individual indicators. This detection/selection process disregards both the covariates and the regressor applied for modelling the indicator. There are 4 different detection functions and you set the one to apply in the JSON file `ai4sh/project_neospectra/process/ml_preprocess/outlier_detection.json`:
- iforest (isolationforest)
- ee (eenvelope/ellipticenvelope)
- lof (lofactor/localoutlierfactor)
- 1csvm (1c-svm/oneclasssvm)

Set "detector": "select" to run all 4 detectors with the same threshold, plotted side by side, and choose which one to apply. The threshold is the expected share of outliers (contamination; nu for 1csvm).

### 1.1.1 Compare and select outlier removal with fixed contamination
Outlier detection can be applied to remove indicator values falling outside a distribution function. Choose 0 = none, 1 = isolation forest (iforest), 2 = elliptic envelope (ee), 3 = local outlier factor(lof), 4 = one class svm (1scvm). The contamination fraction is set in the JSON process file and can not be interactively changed.

suffix: `.olif`, `.olee`, `.ollof` or `.ol1csvm`, for the detector applied (+ optional `#<name_extension>`)

In [ ]:
#%%script false --no-raise-error
process_file = 'project_neospectra/process/ml_preprocess/indicator_outlier_detection_select.json'

structured_process_D, scheme_params_D = Initiate_process(notebook_path, scheme_file, process_file)

if structured_process_D is not None: 
    
    Run_process(structured_process_D, scheme_params_D)

### 1.1.2 Apply predefined outlier removal with interactive contamination

When running the cell below you can interactively change the threshold for a single outlier removal function from its default value in the JSON process file.

When saving the outliers you get 3 options (write answer in full):
- all (all identified outliers removed)
- none (no outlier removed)
- individual (select to remove outliers for individual indicators)

suffix: `.olif`, `.olee`, `.ollof` or `.ol1csvm`, for the detector applied (+ optional `#<name_extension>`)

In [ ]:
#%%script false --no-raise-error
process_file = 'project_neospectra/process/ml_preprocess/indicator_outlier_detection.json'

structured_process_D, scheme_params_D = Initiate_process(notebook_path, scheme_file, process_file)

if structured_process_D is not None: 
    
    Run_process(structured_process_D, scheme_params_D)

## 1.2 Remove indicator values outside a range

Set indicator values outside a [min, max] range to missing (NaN), for example c-org above 10 % when the model is meant for agricultural soils, where a few organic soils would otherwise dominate the model error. Unlike outlier detection nothing is learned from the samples: the limits are fixed and recorded in the preprocessing chain.

The start ranges are set in `data_range` of the JSON process file (`ai4sh/project_neospectra/process/ml_preprocess/indicator_range.json`), in the same format as in `select_indicator_only`; an indicator without an entry starts open. When running the cell the distribution of every indicator is printed (min, 1 %, median, 99 %, max) and plotted with the range limits. Change a range by typing `<indicator> <min> <max>`, e.g. `c-org 0 8`, with `-` for an open end (`ph-water - 8.5`); press Enter to accept or type 0 to skip. Set `"interactive": false` to apply `data_range` as given.

The example runs on the previous dataframe, i.e. after the outlier removal in 1.1; set `"dataframe": "raw"` to run it on the selected indicators.

suffix: `.ir` (+ optional `#<name_extension>`)

In [ ]:
#%%script false --no-raise-error
process_file = 'project_neospectra/process/ml_preprocess/indicator_range.json'

structured_process_D, scheme_params_D = Initiate_process(notebook_path, scheme_file, process_file)

if structured_process_D is not None: 
    
    Run_process(structured_process_D, scheme_params_D)

# 2 Spectral pre-processing

There are many different options for preprocessing spectral data. In many cases it is beneficial to apply a pipeline of pre-processes. The general process steps below, in the recommended order, include:
- trimming and filtering (noise reduction)
- scatter correction (light path and particle size adjustments)
- derivatives (remove baseline shifts and highlight overlapping absorption features)
- variable scaling, including meancentring (eliminates offsets and balances the influence of the bands)
- band reduction (parsimonious model formulation, avoid model over-fitting)
- decomposition (Principal Component Analysis, maximizes variance)

The outlined pipeline is a suggestion and you can reorder the preprocessing steps. Keep in mind that scatter corrections work best on complete spectra (before variable scaling and band reduction), and that derivatives require neighbouring bands (before band reduction).

You define a pipeline of pre-processing steps by setting the "dataframe" JSON parameter of each process to the output (parquet file) of the previous step. If you set "dataframe": "previous" the process takes the latest saved parquet file as input. You can also define a complete pipeline using the process "chemometric_chain" (section 2.7).

## 2.1 Trimming and filtering
Trimming is primarily done when selecting the band range (in e.g. `select_spectra_only.ipynb`) but can also be applied using the multi-filter process. The framework contains 4 single filter options:
 - moving-average
 - gauss 
 - savitzky-golay
 - lowess

You can select any of the 4 filters by running cell 2.1.1, compare custom filters in cell 2.1.2 or apply one predefined filter in cell 2.1.3. The default parameters for each filter are defined in the database, and can be overridden by custom definitions under `ai4sh/project_neospectra/filter`. Check the JSON process files in the cells below to understand how to set a custom filter.

Cell 2.1.4 illustrates how filter length affects LOWESS, and how LOWESS compares with Savitzky-Golay filtering. Cell 2.1.5 is an example on setting a name-extension to a saved filter.

There is a more advanced multi-filter (process: multi_filter_spectra) with an example in cell 2.1.6.

The savitzky-golay filter can also compute derivatives while smoothing - a popular preprocessing for some applications; in the framework you can create this savitzky-golay derivative method by setting the parameter "deriv" to 1 or 2 (the default is 0 - no derivation). See the JSON file defining the savitzky-golay filter, and cell 2.3.4 if you want to run savitzky-golay derivative as a single process.

### 2.1.1 Select filter smoothing

Filter smoothing can be applied to reduce small scale noise due to e.g. sensor noise. The filtering is done on the fly from the dataframe. In the JSON process file you can set either the name of the filter and the system will apply the default filter, or point to a custom filter that will take precedence over the default. The example JSON process file incudes 4 alternatives to choose from: 0 = none, 1 = moving-average, 2 = gauss, 3 = savitzky-golay, 4 = lowess. 

suffix: `.ma<kernel length>`, `.gf<sigma>`, `.sg<window_length>` (`.sg<window_length>.d1` / `.d2` with deriv 1 / 2) or `.lw<n_points>`, e.g. `.ma5`, `.gf1p5`, `.sg7`, for the selected filter (+ optional `#<name_extension>`)

In [ ]:
#%%script false --no-raise-error
process_file = 'project_neospectra/process/ml_preprocess/spectra_filter_select.json'

structured_process_D, scheme_params_D = Initiate_process(notebook_path, scheme_file, process_file)

if structured_process_D is not None:
    
    Run_process(structured_process_D, scheme_params_D)

### 2.1.2 Compare (and select) filters with custom parameters

If you define custom filters with different strengths (parameter settings) you can compare the effects and also select the filter that you identify as the best. It uses the same process as in the previous cell, but with a process file pointing at customised filters. Inspect the JSON process file to understand how it is done.

suffix: `.ma<kernel length>`, `.gf<sigma>`, `.sg<window_length>` (`.sg<window_length>.d1` / `.d2` with deriv 1 / 2) or `.lw<n_points>`, e.g. `.ma5`, `.gf1p5`, `.sg7`, for the selected filter (+ optional `#<name_extension>`)

In [ ]:
#%%script false --no-raise-error
process_file = 'project_neospectra/process/ml_preprocess/spectra_filter_moving-average_compare.json'

structured_process_D, scheme_params_D = Initiate_process(notebook_path, scheme_file, process_file)

if structured_process_D is not None:
    
    Run_process(structured_process_D, scheme_params_D)

### 2.1.3 Apply predefined filter smoothing
Edit the linked JSON process file to apply a predefined filter to the spectra. The default is to apply savitzky-golay filtering as defined in the file `spectra_savitzky-golay_filter.json`, which replicates the database default savitzky-golay filter.

suffix: `.ma<kernel length>`, `.gf<sigma>`, `.sg<window_length>` (`.sg<window_length>.d1` / `.d2` with deriv 1 / 2) or `.lw<n_points>`, e.g. `.ma5`, `.gf1p5`, `.sg7` (+ optional `#<name_extension>`)

In [ ]:
#%%script false --no-raise-error
process_file = 'project_neospectra/process/ml_preprocess/spectra_savitzky-golay_filter.json'

structured_process_D, scheme_params_D = Initiate_process(notebook_path, scheme_file, process_file)

if structured_process_D is not None:
    
    Run_process(structured_process_D, scheme_params_D)

### 2.1.4 LOWESS filter length and comparison with Savitzky-Golay
Compares the default LOWESS filter (n_points:21, with a custom LOWESS filter with n_points:7) and a custom Savitzky-Golay with window_length:7.

In [ ]:
#%%script false --no-raise-error
process_file = 'project_neospectra/process/ml_preprocess/spectra_filter_lowess_vs_sg.json'

structured_process_D, scheme_params_D = Initiate_process(notebook_path, scheme_file, process_file)

if structured_process_D is not None:
    
    Run_process(structured_process_D, scheme_params_D)

### 2.1.5 Moving average with name extension
The JSON process file applies a moving average window of length 5 with the relative weights [1,2,3,2,1] and uses the optional `name_extension` when saving the results (see JSON process file). 

In [ ]:
#%%script false --no-raise-error
process_file = 'project_neospectra/process/ml_preprocess/spectra_filter_moving-average_12321.json'

structured_process_D, scheme_params_D = Initiate_process(notebook_path, scheme_file, process_file)

if structured_process_D is not None:
    
    Run_process(structured_process_D, scheme_params_D)

### 2.1.6 Multi-filter

In a multi-filter you can define different filters with different parameters across different ranges in the input spectra. It is useful for correcting spectra from a particular sensor with noise in particular band regions, or for emulating a sensor with coarser resolutions from a more fine resolution sensor.

The multi-filter is defined in a JSON file (in the example `ai4sh/project_neospectra/filter/multi_filter_neospectra.json`) with parallel lists, one position per region:
- `begin_wavelength` and `end_wavelength` set the regions; the end wavelength is not included, and bands outside every region are dropped from the output;
- for each region, the first filter with a non-zero entry at that position is applied: `moving_average` → `kernel`, `savitzky-golay` → `window_length` (+ `polyorder`), `gauss` → `sigma`, `lowess` → `n_points`;
- an empty kernel or a 0 means "not this filter for this region"; a region without any filter keeps its bands unfiltered;
- `mode` sets how the filters handle the region edges (default "nearest").

The example applies a 5-band moving average to 1350-1440 nm, a savitzky-golay filter (window 5, polyorder 2) to 1450-2390 nm and a gauss filter (sigma 2) to 2400-2500 nm.

suffix: `.mf` (+ optional `#<name_extension>`)

In [ ]:
#%%script false --no-raise-error
process_file = 'project_neospectra/process/ml_preprocess/spectra_multi_filter_example.json'

structured_process_D, scheme_params_D = Initiate_process(notebook_path, scheme_file, process_file)

if structured_process_D is not None:
    
    Run_process(structured_process_D, scheme_params_D)

## 2.2 Scatter correction
There are six different methods for scatter correction included in the framework:
- l1 (each spectrum divided by the sum of its absolute values; normalises total intensity)
- l2 (each spectrum divided by its Euclidean length; normalises spectral energy)
- max (each spectrum divided by its maximum value; peak normalisation)
- snv (Standard Normal Variate; each spectrum centred on its own mean and divided by its own standard deviation)
- msc (Multiplicative Scatter Correction; each spectrum regressed on the mean spectrum, then corrected for the fitted offset and slope)
- detrend (a 2nd-order polynomial fitted to each spectrum is subtracted; removes the curved baseline that snv leaves, normally applied after snv)

All six work per spectrum, removing additive (offset), multiplicative (slope) and, for detrend, curved baseline effects of light scattering.

Not all sequences make sense: snv removes the offset and scale of each spectrum, so l1, l2, max, snv or msc before a later snv have no effect (snv+snv gives the same result as snv). A warning is printed for such a sequence. The classic sequence is snv+detrend.

Cell 2.2.1 compares the scatter correction methods and lets you select any of them, or none. If you want to apply a particular scatter correction method, or sequence of scatter correction methods, use cell 2.2.2.

### 2.2.1 Select scatter correction
Plots raw spectra and all 6 single step scatter corrections (per spectrum); choose 0 = none, 1 = l1, 2 = l2, 3 = max, 4 = snv, 5 = msc, 6 = detrend.

To apply a predefined single scatter correction or sequence of scatter corrections (e.g. snv+detrend) skip this cell and go to cell 2.2.2 and explicitly define the single or sequence of correction(s) to apply.

suffix: `.l1`, `.l2`, `.max`, `.snv`, `.msc` or `.dt`, for the selected scatter correction

In [ ]:
#%%script false --no-raise-error
process_file = 'project_neospectra/process/ml_preprocess/spectra_scatter_correction_select.json'

structured_process_D, scheme_params_D = Initiate_process(notebook_path, scheme_file, process_file)

if structured_process_D is not None: 
    
    Run_process(structured_process_D, scheme_params_D)

### 2.2.2 Apply predefined scatter correction
Apply a predefined scatter correction, defined as a single step or a sequence of two or more scatter corrections.

suffix: the scatter corrections joined, e.g. `.snvdt` for snv+detrend

In [ ]:
#%%script false --no-raise-error
process_file = 'project_neospectra/process/ml_preprocess/spectra_snv+detrend_scatter_correction.json'

structured_process_D, scheme_params_D = Initiate_process(notebook_path, scheme_file, process_file)

if structured_process_D is not None:
    
    Run_process(structured_process_D, scheme_params_D)

## 2.3 Derivatives

If you want to work with derivatives instead of raw data, the framework converts the spectral data per band to derivatives between neighbouring bands. You can set the derivative (1st, 2nd, 3rd etc) in the JSON process file `ai4sh/project_neospectra/process/ml_preprocess/spectra_derivatives.json`. The first derivative (D¹) removes constant baseline offsets (shifts) and identifies the slope changes, turning peaks/valleys into zero-crossings. The second derivative (D²) removes both constant offsets and linear baseline trends. It directly aligns peak minima with the original absorption maxima, sharpening overlapping bands.

There is also a more advanced function that allows you to define derivatives for specific pairs of bands (TO BE COMPLETED) and generates a dataframe with only the derivatives between the specified pairs.

### 2.3.1 Continuous derivatives

suffix: `.d1`, `.d2` ... for the derivative set (`.d1app` etc. if the derivatives are appended to the bands)

In [ ]:
#%%script false --no-raise-error
process_file = 'project_neospectra/process/ml_preprocess/spectra_derivatives.json'

structured_process_D, scheme_params_D = Initiate_process(notebook_path, scheme_file, process_file)

if structured_process_D is not None:
    
    Run_process(structured_process_D, scheme_params_D)

### 2.3.2 Discrete derivatives

TO BE COMPLETED

suffix: TO BE COMPLETED

In [ ]:
%%script false --no-raise-error
process_file = 'project_neospectra/process/ml_preprocess/spectra_discrete_derivatives.json'

structured_process_D, scheme_params_D = Initiate_process(notebook_path, scheme_file, process_file)

if structured_process_D is not None:
    
    Run_process(structured_process_D, scheme_params_D)

### 2.3.3 Compare derivatives
The derivative is applied to every dataframe (parquet file) listed in "dataframe_array" parameter of the JSON process file. The list should represent different levels of preprocessing (e.g. raw, filtered, scaled and scatter corrected). The results are plotted together to visually explore the difference; choose 0 = none or the number of the input whose derivative to save.

suffix: `.d1`, `.d2` ... added to the selected input dataframe

In [ ]:
#%%script false --no-raise-error
process_file = 'project_neospectra/process/ml_preprocess/spectra_derivatives_compare_select.json'

structured_process_D, scheme_params_D = Initiate_process(notebook_path, scheme_file, process_file)

if structured_process_D is not None:
    
    Run_process(structured_process_D, scheme_params_D)

### 2.3.4 Savitzky-Golay derivative

The Savitzky-Golay derivative method integrates Savitzky-Golay filtering and derivation in a single process. This is a common approach for correcting diffuse reflectance data for Near-InfraRed (NIR) soil. In a pipeline, apply it after scatter correction and before variable scaling (raw → scatter correction → Savitzky-Golay derivative → meancentring → band reduction → PCA); it replaces both the filter (2.1) and the derivative (2.3.1) steps. Do not apply a scatter correction (snv, msc) after a derivative: scatter corrections assume a spectrum with its baseline and slope, which the derivative has removed. Cell 2.3.4 just runs the Savitzky-Golay derivative method on the raw input data as an example; set "dataframe" to a scatter-corrected file (e.g. `1350-2500_10.snv`) to follow the recommended order. Cell 2.7.2 is a complete pipeline definition in a single JSON process file and has the Savitzky-Golay derivative method as one of the steps.

suffix: `.sg<window_length>.d1` (`.d2` for deriv 2), e.g. `.sg5.d1` (+ optional `#<name_extension>`)

In [ ]:
#%%script false --no-raise-error
process_file = 'project_neospectra/process/ml_preprocess/spectra_savitzky-golay_derivative.json'

structured_process_D, scheme_params_D = Initiate_process(notebook_path, scheme_file, process_file)

if structured_process_D is not None:
    
    Run_process(structured_process_D, scheme_params_D)

## 2.4 Variable scaling (including meancentring)
Variable scaling works per band (variable) across all samples. Meancentring shifts the focus of the dataset from the absolute values to the variation around the average. It acts as a standalone technique and sometimes as the baseline first step for the other scaling functions. The variable scaling methods available in the framework include:
- meancentring
- autoscaling
- paretoscaling
- poissonscaling

Cell 2.4.1 compares the variable scaling methods and lets you select any of them, or none. If you want to apply a particular variable scaling method, use cell 2.4.2.

### 2.4.1 Select variable scaling
Plots raw spectra and all 4 variable (band-wise) scalings; choose 0 = none, 1 = meancentring, 2 = autoscaling, 3 = paretoscaling, 4 = poissonscaling. 

If you want to apply one specific variable scaling (e.g. meancentring, that is commonly used as a default pre-processing step), you can skip this cell and just apply the next cell (2.4.2).

suffix: `.mc` (meancentring), `.as` (autoscaling), `.ps` (paretoscaling) or `.poi` (poissonscaling), for the selected variable scaling

In [ ]:
#%%script false --no-raise-error
process_file = 'project_neospectra/process/ml_preprocess/spectra_variable_scaling_select.json'

structured_process_D, scheme_params_D = Initiate_process(notebook_path, scheme_file, process_file)

if structured_process_D is not None: 
    
    Run_process(structured_process_D, scheme_params_D)

### 2.4.2 Apply predefined variable scaling
Edit the linked JSON process file to apply a predefined variable scaling to the spectra. The default is to apply meancentring.

suffix: `.mc` (meancentring), `.as` (autoscaling), `.ps` (paretoscaling) or `.poi` (poissonscaling)

In [ ]:
#%%script false --no-raise-error
process_file = 'project_neospectra/process/ml_preprocess/spectra_meancentring.json'

structured_process_D, scheme_params_D = Initiate_process(notebook_path, scheme_file, process_file)

if structured_process_D is not None: 
    
    Run_process(structured_process_D, scheme_params_D)

## 2.5 Band reduction
There are 2 options for removing individual spectral bands without considering either the target feature or the regressor applied in the modelling. The objective is to reduce the input for the regression modelling and to avoid model over-fitting. The two options are:
- variance threshold
- agglomeration 

### 2.5.1 Variance select spectral bands to keep/discard 
The band variances are computed after minmax scaling: each band is scaled to the range 0-1 across all samples, so that bands with high and low reflectance are compared on the same scale. (Standard scaling would give every band the variance 1, and robust scaling nearly constant variances, so neither can separate bands.)

On raw spectra the spread of the scaled band variances is narrow and the threshold separates bands only weakly; the variance threshold is more informative after scatter correction.

When running the cell, the first plot shows the scaled variance of every band, and the band variance minimum, median and maximum are printed. You then interactively set the threshold; bands with a variance at or below the threshold are discarded. At the prompt you can:
- enter a threshold (between the minimum and the maximum band variance) - the plot is redrawn with the discarded bands marked and the number of retained and discarded bands printed; you can enter new thresholds until you are satisfied;
- press Enter to accept the latest threshold and save the retained bands;
- enter 0 (or any value below the minimum band variance) to skip - nothing is saved.

suffix: `.vt` + threshold without the leading "0.", e.g. `.vt028` for 0.028 (+ optional `#<name_extension>`)

In [ ]:
#%%script false --no-raise-error
process_file = 'project_neospectra/process/ml_preprocess/spectra_variance_threshold_selection.json'

structured_process_D, scheme_params_D = Initiate_process(notebook_path, scheme_file, process_file)

if structured_process_D is not None: 
    
    Run_process(structured_process_D, scheme_params_D)

### 2.5.2 Agglomerate similar bands with ward clustering
The ward clustering can be set to either agglomerate only neighbouring bands ("connectivity": true), which is the default and recommended setting, or to cluster similar bands across the entire spectrum. The selection setting is defined in the JSON file `./project_neospectra/agglomeration/ward_clustering_neospectra.json` that you can edit before running the cell below. When running the cell you will also get the option to select the final number of clusters to save.

suffix: `.wc<n_clusters>`, e.g. `.wc40` (+ optional `#<name_extension>`)

In [ ]:
#%%script false --no-raise-error
process_file = 'project_neospectra/process/ml_preprocess/spectra_ward_clustering.json'

structured_process_D, scheme_params_D = Initiate_process(notebook_path, scheme_file, process_file)

if structured_process_D is not None:
    
    Run_process(structured_process_D, scheme_params_D)

## 2.6 Decomposition

### 2.6.1 Principal Component Analysis (PCA) Decomposition
Apply the decomposition (PCA) defined in the linked JSON process file to one input dataframe.

suffix: `.pca5` (`.pca` + the number of components)

In [ ]:
#%%script false --no-raise-error
process_file = 'project_neospectra/process/ml_preprocess/spectra_decomposition.json'

structured_process_D, scheme_params_D = Initiate_process(notebook_path, scheme_file, process_file)

if structured_process_D is not None:
    
    Run_process(structured_process_D, scheme_params_D)

### 2.6.2 Compare PCA decompositions
The decomposition is applied to every dataframe (parquet file) listed in "dataframe_array" parameter of the JSON process file (e.g. raw and the outputs of different preprocessing pipelines). The scores and the share of spectral variance explained are plotted for each input; choose 0 = none or the number of the input to save.

suffix: `.pca5` (`.pca` + the number of components) added to the selected input dataframe

In [ ]:
#%%script false --no-raise-error
process_file = 'project_neospectra/process/ml_preprocess/spectra_decomposition_compare_select.json'

structured_process_D, scheme_params_D = Initiate_process(notebook_path, scheme_file, process_file)

if structured_process_D is not None:
    
    Run_process(structured_process_D, scheme_params_D)

### 2.6.3 Wavelet denoise
Discrete wavelet transform (DWT, `wavelet` db4, `level` 3) of every spectrum; the detail coefficients are soft-thresholded (universal threshold, noise from the finest details) and the spectrum is reconstructed. Same bands, less high-frequency noise; also available as the `wavelet` item of a chemometric chain.

suffix: `.wd`

In [ ]:
#%%script false --no-raise-error
process_file = 'project_neospectra/process/ml_preprocess/spectra_wavelet_denoise.json'

structured_process_D, scheme_params_D = Initiate_process(notebook_path, scheme_file, process_file)

if structured_process_D is not None:
    
    Run_process(structured_process_D, scheme_params_D)

### 2.6.4 Wavelet coefficients
The approximation coefficients at `level` (plus `details` detail levels, coarsest first) replace the bands as features `wt1 .. wtn` — a compression of the spectrum like PCA, but local: each coefficient covers a wavelength region. Band selection on compiled dataframes (`ml_preprocess_compiled.ipynb`) can then select coefficients.

suffix: `.wt`

In [ ]:
#%%script false --no-raise-error
process_file = 'project_neospectra/process/ml_preprocess/spectra_wavelet_coefficients.json'

structured_process_D, scheme_params_D = Initiate_process(notebook_path, scheme_file, process_file)

if structured_process_D is not None:
    
    Run_process(structured_process_D, scheme_params_D)

## 2.7 Chemometric chain

The process "chemometric_chain" runs a complete pipeline of pre-processing steps in one go, defined in the parameter "chemometrics_array". The steps are applied in the listed order, each step taking the output of the step before. All steps are plotted together and the result is saved after you accept it. Only the final dataframe is saved, with the same name and the same pre-processing chain (companion JSON) as if you had run the steps one by one.

Each item in "chemometrics_array" is either a method name, applied with its default settings, or an object with custom settings:

| Step | Method name (default settings) | Object (custom settings) |
|---|---|---|
| Filter | `moving-average`, `gauss`, `savitzky-golay`, `lowess` | `{"filter": "<parameter set name or JSON file>"}` |
| Multi-filter | - | `{"multi_filter": "<JSON file>"}` |
| Scatter correction | `l1`, `l2`, `max`, `snv`, `msc`, `detrend` | `{"scatter_correction": ["snv", "detrend"]}` |
| Derivative | `derivative` (1st) | `{"derivative": 2}` |
| Variable scaling | `meancentring`, `autoscaling`, `paretoscaling`, `poissonscaling` | `{"scaling": "autoscaling"}` |
| Variance threshold | - | `{"variance_threshold": {"threshold": 0.03}}` |
| Ward clustering | - | `{"ward_clustering": {"n_clusters": 40, "connectivity": true}}` |
| PCA | `pca` (5 components) | `{"pca": 5}` |

Variance threshold and ward clustering have no default settings: the threshold and the number of clusters must be given. The chain is not interactive; use the single pre-process cells above to find the settings (e.g. the threshold or the number of clusters) and then define them in the chain. The same syntax is used for "chemometrics_array" in the spectra plot process.

### 2.7.1 Simple pre-processing chain

Cell 2.7 runs a simple chain: snv, derivative, meancentring and pca.

suffix: the suffixes of the steps in the chain, e.g. `.snv.d1.mc.pca5` (+ optional `#<name_extension>`)

In [ ]:
#%%script false --no-raise-error
process_file = 'project_neospectra/process/ml_preprocess/chemometric_default_chain.json'

structured_process_D, scheme_params_D = Initiate_process(notebook_path, scheme_file, process_file)

if structured_process_D is not None:
    
    Run_process(structured_process_D, scheme_params_D)

### 2.7.2 Complete pre-processing pipeline

The pre-processing pipeline in cell 2.7.2 is an example of a complete processing chain in a single JSON process file, following the recommended order:

raw -> scatter correction (snv) -> Savitzky-Golay derivative (D1) -> meancentring -> ward clustering (40 clusters) -> PCA (5 components)

```
"chemometrics_array": [
  "snv",
  {"filter": "./project_neospectra/filter/savitzky-golay_d1.json"},
  "meancentring",
  {"ward_clustering": {"n_clusters": 40, "connectivity": true}},
  {"pca": 5}
]
```

suffix: `.snv.sg5.d1.mc.wc40.pca5`

In [ ]:
#%%script false --no-raise-error
process_file = 'project_neospectra/process/ml_preprocess/chemometric_pipeline_snv+sgd1+mc+wc+pca.json'

structured_process_D, scheme_params_D = Initiate_process(notebook_path, scheme_file, process_file)

if structured_process_D is not None:
    
    Run_process(structured_process_D, scheme_params_D)

### 2.7.3 Load existing pre-processing chain

Instead of writing your own pre-processing chain you can point to an existing chain and run that; it requires that your input dataframe has the same spectral range and resolution as the chain you load (this is checked before running).

Set the parameter "chain_from" (instead of "chemometrics_array") to a saved pre-processed dataframe: the chain recorded in its companion JSON is run on the input dataframe, step by step with the recorded settings (e.g. the filter settings and the number of ward clusters). Give "chain_from" as:
- the name of a dataframe in "project_root_fp" (.json optional), e.g. "1350-2500_10.snv.sg5.d1.mc.wc40.pca5";
- or a path relative to the project root, to use a chain from another dataset, e.g. "./project_neospectra/data/spectra/neospectra/ds2/ai4sh/1350-2500_10.snv.sg5.d1.mc.wc40.pca5.json".

The steps are fitted on the new input: e.g. the variance threshold, the ward clusters and the PCA are calculated from the new dataframe with the same settings. To predict independent samples with a model trained on a pre-processed dataframe, use the saved model instead (notebook `ml_predict`), which applies the chain as fitted on the training samples.

The example in cell 2.7.3 reruns the chain of cell 2.7.2 on the same raw dataframe; "name_extension": "rerun" keeps the result apart from the original.

suffix: the suffixes of the loaded chain, e.g. `.snv.sg5.d1.mc.wc40.pca5` (+ optional `#<name_extension>`)

In [ ]:
#%%script false --no-raise-error
process_file = 'project_neospectra/process/ml_preprocess/chemometric_pipeline_load.json'

structured_process_D, scheme_params_D = Initiate_process(notebook_path, scheme_file, process_file)

if structured_process_D is not None:
    
    Run_process(structured_process_D, scheme_params_D)

# 3 Delete redundant dataframes

Evaluating different pre-processing options leaves many dataframes that are not needed once you have chosen your final pre-processing pipeline(s). The process "delete_dataframes" deletes the pre-processed dataframes (parquet + companion JSON) that you do not keep and removes them from the `.previous_dataframe` history.

Set the dataframes to keep in "keep_array" (default ["raw", "previous"]):
- raw - the selected (not pre-processed) datasets; these are never deleted anyway;
- previous - the dataframe(s) of the latest saved pre-process;
- the names of the pre-processed dataframes to keep (.parquet optional), e.g. "1350-2500_10.snv.sg5.d1.mc.wc40.pca5".

A pipeline output only depends on the raw dataset (the companion JSON records the complete chain, and modelling refits it from the raw dataset), so the intermediate dataframes of a kept pipeline can be deleted. The dataframes to delete are listed before you confirm with y/n, and a name in "keep_array" that does not exist stops the process before anything is deleted. Plots and model results are not deleted.

In [ ]:
#%%script false --no-raise-error
process_file = 'project_neospectra/process/ml_preprocess/spectra_delete_dataframes.json'

structured_process_D, scheme_params_D = Initiate_process(notebook_path, scheme_file, process_file)

if structured_process_D is not None:
    
    Run_process(structured_process_D, scheme_params_D)